# Hipótesis 1

 Factores de entorno: La iluminación (P43_1B), la vigilancia en zonas de obra (P43_1G) y la señalización (P43_1A) son los factores más asociados con la percepción de empeoramiento de seguridad.

In [3]:
# =============================================================================
# 0. CONFIGURACIÓN
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Logit
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

DATA_DIR = './datos_limpios'
corredores = pd.read_csv(f'{DATA_DIR}/corredores_limpio.csv')

# 1. Selección de Variables

In [4]:
# =============================================================================
# 1. SELECCIÓN DE VARIABLES
# =============================================================================
cols_p43_1 = [c for c in corredores.columns if c.startswith('P43_1')]

vars_h3 = ['seguridad_empeoro'] + cols_p43_1 + ['obras_iniciadas', 'sector', 'tamano_empresa', 'tramo', 'P1', 'indice_satisfaccion']

df = corredores[vars_h3].copy()

# Nombres descriptivos (para visualizaciones y reportes)
labels = {
    'seguridad_empeoro'  : 'Percepción de seguridad empeoró',
    'P43_1A'           : 'Señalización',
    'P43_1B'           : 'Iluminación',
    'P43_1C'           : 'Presencia de personas',
    'P43_1D'           : 'Presencia policial',
    'P43_1E'           : 'Delincuencia',
    'P43_1F'           : 'Condiciones de infraestructura',
    'P43_1G'           : 'Vigilancia en zonas de obra',
    'obras_iniciadas'  : 'Obras iniciadas en el tramo (dummy)',
    'sector'           : 'Sector productivo',
    'tamano_empresa'   : 'Tamaño empresa',
    'tramo'            : 'Tramo geográfico',
    'P1'               : 'Número de empleados',
    'indice_satisfaccion': 'Índice de satisfacción con el entorno',
}

print(f"Dimensiones: {df.shape}")
print(f"\nVariables y descripciones:")
for col in df.columns:
    print(f"  {col:<20} → {labels.get(col, '—')}")
df.head()

Dimensiones: (2105, 14)

Variables y descripciones:
  seguridad_empeoro    → Percepción de seguridad empeoró
  P43_1A               → Señalización
  P43_1B               → Iluminación
  P43_1C               → Presencia de personas
  P43_1D               → Presencia policial
  P43_1E               → Delincuencia
  P43_1F               → Condiciones de infraestructura
  P43_1G               → Vigilancia en zonas de obra
  obras_iniciadas      → Obras iniciadas en el tramo (dummy)
  sector               → Sector productivo
  tamano_empresa       → Tamaño empresa
  tramo                → Tramo geográfico
  P1                   → Número de empleados
  indice_satisfaccion  → Índice de satisfacción con el entorno


,seguridad_empeoro,P43_1A,P43_1B,P43_1C,P43_1D,P43_1E,P43_1F,P43_1G,obras_iniciadas,sector,tamano_empresa,tramo,P1,indice_satisfaccion
0,1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0,Servicio,Grande,18,3.0,3.250000
1,0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0,NaN,Pequeña,18,1.0,1.500000
2,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,Servicio,Grande,18,1.0,1.000000
3,0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,Comercio,Micro,18,5.0,2.833333
4,0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0,Servicio,Micro,18,4.0,2.333333


# 2. One Hot Encoding

Realizamos el proceso a las categóricas: sector, tamano_empresa y tramo

In [5]:
# =============================================================================
# 2. ONE HOT ENCODING
# =============================================================================

df_encoded = df.copy()

# --- sector: OHE eliminando 'Comercio' como categoría de referencia ---
df_encoded['sector'] = df_encoded['sector'].replace('NS/NR', np.nan)
ohe_sector = pd.get_dummies(df_encoded['sector'], prefix='sector', dummy_na=False)
ohe_sector = ohe_sector.drop(columns=['sector_Comercio'], errors='ignore')

# --- tamano_empresa: OHE eliminando 'Micro' como categoría de referencia ---
df_encoded['tamano_empresa'] = df_encoded['tamano_empresa'].replace('NS/NR', np.nan)
ohe_tamano = pd.get_dummies(df_encoded['tamano_empresa'], prefix='tamano_empresa', dummy_na=False)
ohe_tamano = ohe_tamano.drop(columns=['tamano_empresa_Micro'], errors='ignore')

# --- tramo: OHE eliminando tramo 1 como categoría de referencia ---
ohe_tramo = pd.get_dummies(df_encoded['tramo'], prefix='tramo', dummy_na=False)
ohe_tramo = ohe_tramo.drop(columns=['tramo_1'], errors='ignore')

# Reemplazar columnas originales por sus dummies
df_encoded = pd.concat([
    df_encoded.drop(columns=['sector', 'tamano_empresa', 'tramo']),
    ohe_sector,
    ohe_tamano,
    ohe_tramo
], axis=1)

print("One Hot Encoding completado — k-1 variables por categoría:")
print(f"  sector        : {list(ohe_sector.columns)}")
print(f"  tamano_empresa: {list(ohe_tamano.columns)}")
print(f"  tramo         : {list(ohe_tramo.columns[:5])} ... ({ohe_tramo.shape[1]} dummies)")
print(f"\nDimensiones finales: {df_encoded.shape}")
df_encoded.head()

One Hot Encoding completado — k-1 variables por categoría:
  sector        : ['sector_Industria', 'sector_Servicio']
  tamano_empresa: ['tamano_empresa_Grande', 'tamano_empresa_Mediana', 'tamano_empresa_Pequeña']
  tramo         : ['tramo_2', 'tramo_3', 'tramo_4', 'tramo_5', 'tramo_6'] ... (18 dummies)

Dimensiones finales: (2105, 34)


,seguridad_empeoro,P43_1A,P43_1B,P43_1C,P43_1D,P43_1E,P43_1F,P43_1G,obras_iniciadas,P1,indice_satisfaccion,sector_Industria,sector_Servicio,tamano_empresa_Grande,tamano_empresa_Mediana,tamano_empresa_Pequeña,tramo_2,tramo_3,tramo_4,tramo_5,tramo_6,tramo_7,tramo_8,tramo_9,tramo_10,tramo_11,tramo_12,tramo_13,tramo_14,tramo_15,tramo_16,tramo_17,tramo_18,tramo_19
0,1,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0,3.0,3.250000,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0,1.0,1.500000,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,1.0,1.000000,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,5.0,2.833333,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
4,0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0,4.0,2.333333,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
